# Exam 3

**Name:** Prinsa Ghimire  
**Course:** CPSMA 4313

In [25]:
import pandas as pd
import numpy as np
import re

## 1(a) Store the data as a pandas dataframe. Examine each datatype and comment on the appropriateness of
each

In [26]:
url = "https://raw.githubusercontent.com/nurfnick/Data_Viz/refs/heads/main/Data_Sets/Activity_Dataset_V1.csv"

df = pd.read_csv(url)

df.head()

,Unnamed: 0,activity_day,workout_type,distance,time,calories,total_steps,avg_speed,avg_cadence,max_cadence,...,max_pace,min_pace,avg_heart_rate,max_heart_rate,min_heart_rate,vo2_max(%),aerobic(%),anaerobic(%),intensive(%),light(%)
0,0,2022-01-01,Freestyle,9.30,77,123,NaN,18.88,168.54,138.30,...,NaN,NaN,112.5,122.0,103,19,28,2,7,50
1,1,2022-01-01,Freestyle,3.44,96,55,NaN,29.65,125.92,292.81,...,NaN,NaN,111.0,122.0,100,42,28,2,29,88
2,2,2022-01-01,Indoor Cycling,6.34,85,33,NaN,17.85,81.93,323.69,...,NaN,NaN,95.0,90.0,100,1,32,0,22,43
3,3,2022-01-01,Walking,7.91,42,82,1571.0,22.10,29.63,180.16,...,28:58,07:58,83.0,85.0,81,3,22,0,24,65
4,4,2022-01-01,Open Water,8.99,36,131,NaN,25.83,64.55,342.89,...,NaN,NaN,138.0,166.0,110,7,0,5,21,88


In [27]:
df.dtypes

Unnamed: 0          int64
activity_day       object
workout_type       object
distance          float64
time                int64
calories            int64
total_steps       float64
avg_speed         float64
avg_cadence       float64
max_cadence       float64
avg_pace           object
max_pace           object
min_pace           object
avg_heart_rate    float64
max_heart_rate    float64
min_heart_rate      int64
vo2_max(%)          int64
aerobic(%)          int64
anaerobic(%)        int64
intensive(%)        int64
light(%)            int64
dtype: object

The dataset contains object, integer, and float datatypes. `activity_day` should be datetime (time-aware analysis), and `avg_pace` starts as text because it is stored as `mm:ss`. Numeric workout variables such as `calories`, `aerobic`, and `max_cadence` are appropriate as numeric types for aggregation and comparisons.

## 1(b) Remove the column that repeats the indexes and is ‘unnamed’ as a column.

In [28]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

df.head()

,activity_day,workout_type,distance,time,calories,total_steps,avg_speed,avg_cadence,max_cadence,avg_pace,max_pace,min_pace,avg_heart_rate,max_heart_rate,min_heart_rate,vo2_max(%),aerobic(%),anaerobic(%),intensive(%),light(%)
0,2022-01-01,Freestyle,9.30,77,123,NaN,18.88,168.54,138.30,NaN,NaN,NaN,112.5,122.0,103,19,28,2,7,50
1,2022-01-01,Freestyle,3.44,96,55,NaN,29.65,125.92,292.81,NaN,NaN,NaN,111.0,122.0,100,42,28,2,29,88
2,2022-01-01,Indoor Cycling,6.34,85,33,NaN,17.85,81.93,323.69,NaN,NaN,NaN,95.0,90.0,100,1,32,0,22,43
3,2022-01-01,Walking,7.91,42,82,1571.0,22.10,29.63,180.16,07:58,28:58,07:58,83.0,85.0,81,3,22,0,24,65
4,2022-01-01,Open Water,8.99,36,131,NaN,25.83,64.55,342.89,NaN,NaN,NaN,138.0,166.0,110,7,0,5,21,88


I removed the extra column whose name starts with `Unnamed` because it is just a repeated index created during file export/import. Keeping it would add redundant information and can confuse later analysis. After dropping it, the dataframe contains only meaningful variables for the exam tasks.

## 1(c) Clean Column Names Using Regular Expressions

In [29]:
df.columns = [re.sub(r'\s*\([^)]*\)', '', col).strip().lower().replace(' ', '_') for col in df.columns]

df.columns

Index(['activity_day', 'workout_type', 'distance', 'time', 'calories',
       'total_steps', 'avg_speed', 'avg_cadence', 'max_cadence', 'avg_pace',
       'max_pace', 'min_pace', 'avg_heart_rate', 'max_heart_rate',
       'min_heart_rate', 'vo2_max', 'aerobic', 'anaerobic', 'intensive',
       'light'],
      dtype='object')

I cleaned column names using a regular expression so the solution follows the exam requirement. The regex removes the parenthetical unit text (such as `(%)`), then I trim trailing spaces with `.strip()`, convert to lowercase, and replace spaces with underscores. This creates consistent, analysis-friendly column names.

## 1(d) Convert ‘activity day’ column into a datetime format

In [30]:
df['activity_day'] = pd.to_datetime(df['activity_day'])

df['activity_day'].head()

0   2022-01-01
1   2022-01-01
2   2022-01-01
3   2022-01-01
4   2022-01-01
Name: activity_day, dtype: datetime64[ns]

I converted `activity_day` from text (`object`) into datetime using `pd.to_datetime(...)`. This is important because datetime values support valid time-based analysis, including weekday extraction, date sorting, and date filtering used later in the exam.

## 1(e) Impute total_steps

In [31]:
df['total_steps'] = df['total_steps'].fillna(df['total_steps'].median())

df['total_steps'] = df['total_steps'].astype(int)

df['total_steps'].head()

0    3444
1    3444
2    3444
3    1571
4    3444
Name: total_steps, dtype: int32

Median imputation was used because it is less affected by outliers than the mean. After filling missing values, converting `total_steps` to integer is appropriate since step counts are whole numbers.

## 1(f) Convert avg_pace to Float

In [32]:
# Keep original pace text, then create float minutes representation.
df['avg_pace_original'] = df['avg_pace']

def pace_to_float(pace):
    if pd.isna(pace) or str(pace).strip() == '':
        return np.nan

    parts = str(pace).split(':')

    if len(parts) == 2:
        minutes = int(parts[0])
        seconds = int(parts[1])
        return minutes + seconds / 60

    return np.nan

# Converted pace used for analysis in decimal minutes.
df['avg_pace'] = df['avg_pace'].apply(pace_to_float)

df[['avg_pace_original', 'avg_pace']].head()

,avg_pace_original,avg_pace
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,07:58,7.966667
4,NaN,NaN


The non-empty pace values were converted from `mm:ss` into decimal minutes (for example, `3:30` becomes `3.5`). I kept the original text pace in `avg_pace_original` and replaced `avg_pace` with the float representation for analysis.

## 1(g) Group by workout_type and summarize calories

In [33]:
group_stats = df.groupby('workout_type')['calories'].agg(['mean', 'median', 'count', 'std'])

group_stats

,mean,median,count,std
workout_type,,,,
Cricket,307.548387,330.0,93,149.950609
Freestyle,278.552083,294.0,96,163.703768
Indoor Cycling,280.450000,306.0,80,156.633322
Open Water,296.747253,328.0,91,160.068499
Outdoor Cycling,299.129412,301.0,85,158.731263
Outdoor Running,301.469136,349.0,81,165.725080
Pool Swimming,283.414894,300.0,94,157.576703
Trail Run,267.966667,264.0,90,155.748533
Treadmill,278.142857,269.5,98,146.963352


This table compares calorie behavior by workout type. `mean` and `median` show central tendency, `count` shows how many workouts are in each category, and `std` shows variability (higher values indicate less consistency).

## 1(h) Create Indicator Column for Aerobic >= 30%

In [34]:
df['aerobic_30_or_more'] = df['aerobic'] >= 30

df[['aerobic', 'aerobic_30_or_more']].head()

,aerobic,aerobic_30_or_more
0,28,False
1,28,False
2,32,True
3,22,False
4,0,False


The indicator column returns `True` when aerobic activity is at least 30%. This creates a simple binary feature that can be used to count or compare workouts meeting the threshold.

## 1(i) Find Day and Workout Type with Maximum Max Cadence

In [35]:
df['day_of_week'] = df['activity_day'].dt.day_name()

max_row = df.loc[df['max_cadence'].idxmax(), ['day_of_week', 'workout_type', 'max_cadence']]

max_row

day_of_week         Wednesday
workout_type    Pool Swimming
max_cadence            349.78
Name: 726, dtype: object

The output identifies the weekday and workout type associated with the maximum observed `max_cadence`. This directly answers which day-workout combination produced the peak cadence value in the dataset.

# Resources Used

1. https://raw.githubusercontent.com/nurfnick/Data-Viz/refs/heads/main/Data%20Sets/Activity%20Dataset%20V1.csv
2. https://github.com/nurfnick/Data-Viz
3. https://pandas.pydata.org/docs/
4. https://numpy.org/doc/
5. https://docs.python.org/3/library/re.html
6. https://docs.python.org/3/library/datetime.html
7. https://stackoverflow.com/
8. https://www.kaggle.com/

"I attest that the resources above were the only ones utilized in completing the exam and the work included is my own and no one else from the course."

## LLM Queries Used (exact prompts)
1. "How do I remove an Unnamed index column in pandas after reading a CSV?"
2. "How can I use regex with pandas to remove text inside parentheses in column names?"
3. "How do I convert a pandas column like mm:ss pace into decimal minutes as float?"
4. "How do I get the day name (Monday, Tuesday, etc.) from a datetime column in pandas?"
5. "How do I group by workout type and calculate mean, median, count, and std of calories?"

## Accepted / Rejected Responses
- Accepted: regex-based column cleaning with `re.sub(r'\s*\([^)]*\)', '', col)` because it explicitly requires regular expressions.
- Accepted: converting `activity_day` using `pd.to_datetime(...)`, then deriving weekday with `.dt.day_name()`.
- Accepted: converting pace from `mm:ss` to decimal minutes by splitting on `:` and using `minutes + seconds/60`.
- Rejected: manually renaming each column one-by-one because it does not satisfy the regex requirement in part (c).
- Rejected: replacing missing `total_steps` with 0 because it can bias summaries downward; median is more appropriate for robust imputation.